# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'code repository', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [3]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
5 days ago
•
1.37M
•
1.33k
zai-org/GLM-5.2
Updated
1 day ago
•
191k
•
3.32k
baidu/Unlimited-OCR
Updated
about 11 hours ago
•
885k
•
1.68k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
8 days ago
•
323k
•
673
deepseek-ai/DeepSeek-V4-Pro-DSpark
Updated
6 days ago
•
9.

In [4]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [5]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [11]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n6 days ago\n•\n1.46M\n•\n1.43k\nzai-org/GLM-5.2\nUpdated\n2 days ago\n•

In [13]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [14]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the premier AI community and collaboration platform shaping the future of machine learning. At its core, Hugging Face provides the largest ecosystem where ML engineers, researchers, and enthusiasts come together to create, share, and discover cutting-edge machine learning models, datasets, and applications.

The platform hosts over **2 million models**, **500k+ datasets**, and **1 million+ applications**—making it the central hub for open-source machine learning innovation. Hugging Face empowers the next generation of machine learning practitioners to learn, experiment, and build the future of AI together with an open and ethical approach.

---

## Platform & Solutions

- **Models:** Explore and deploy millions of pre-trained models spanning NLP, computer vision, speech, and more.
- **Datasets:** Access a vast repository of curated datasets to accelerate data science and AI projects.
- **Spaces:** Share and discover ML-powered applications and demos created by the community.
- **HuggingChat:** A conversational AI solution for seamless interaction with AI models.
- **Inference Endpoints & Providers:** Scalable AI model hosting and inference solutions for enterprises.
- **Storage Buckets:** Flexible, secure storage solutions optimized for ML workflows.
- **Enterprise Plans & Hugging Face PRO:** Enterprise-grade tools, premium support, and dedicated AI infrastructure tailored for organizational needs.

---

## Company Culture

Hugging Face is deeply rooted in community and openness. The company fosters a collaborative environment where sharing knowledge and building together is paramount. Their open platform encourages transparency, ethical AI, and innovation driven by collective intelligence.

Here, engineers and scientists work at the edge of AI research, supported by a vibrant and ever-growing community. Hugging Face is not just a company but a movement advocating for an accessible and responsible AI future.

---

## Customers & Community

Hugging Face serves a diverse range of customers from individual developers and academic researchers to large-scale enterprises seeking to integrate state-of-the-art AI solutions. Their platform's open ecosystem attracts:
- Machine learning engineers and data scientists
- Academic and research institutions
- AI startups and industry leaders
- Enterprises needing AI model deployment and support

The company thrives on its engaged community present across GitHub, Discord, forums, and social media, continuously contributing models, datasets, tutorials, and discussions.

---

## Careers & Opportunities

Hugging Face offers rewarding career opportunities for individuals passionate about machine learning, open source, and community-driven technology:
- AI Research Scientists
- Software Engineers (Backend, Frontend, DevOps)
- Machine Learning Engineers
- Community Managers and Developer Advocates
- Enterprise Solutions Architects and Sales

Joining Hugging Face means being part of a global AI revolution alongside talented peers in a culture that values innovation, openness, and impact.

---

## Brand Identity

- **Colors:** Bright, warm yellows (#FFD21E, #FF9D00) balanced with neutral grays (#6B7280), reflecting energy and trust.
- **Logo & Assets:** Available in multiple formats (.svg, .png, .ai) for flexible usage.

---

## Connect & Learn More

- Website: [huggingface.co](https://huggingface.co)  
- Community Channels: Discord, GitHub, Forum  
- Social Media: Twitter | LinkedIn  
- Blog & Learning Resources: Regular posts, tutorials, daily research papers

---

**Hugging Face** — Building the future of AI, together.  
Join the community, explore cutting-edge tools, and transform your machine learning projects today!

In [15]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [16]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face
Hugging Face is a pioneering collaborative platform dedicated to advancing machine learning and artificial intelligence. As the home of a thriving AI community, Hugging Face enables researchers, developers, and organizations worldwide to create, discover, and share cutting-edge machine learning models, datasets, and applications.

The platform hosts over **2 million models**, **500,000+ datasets**, and **more than 1 million AI applications**, making it the most comprehensive repository and collaboration hub for AI development today.

---

## What We Offer

### Models, Datasets & Spaces
- **Models:** Access and contribute to a vast library including state-of-the-art models like language transformers, vision systems, and speech recognition.
- **Datasets:** Explore extensive datasets suited for training, testing, and benchmarking ML applications.
- **Spaces:** Run and showcase ML applications with interactive demos, facilitating real-time collaboration and innovation.

### AI Applications & Tools
- **HuggingChat:** An advanced conversational AI experience.
- **Inference Providers & Endpoints:** Seamless deployment and integration of AI models.
- **Storage Buckets:** Scalable infrastructure for hosting datasets and models.

---

## Our Community & Culture
At Hugging Face, community lies at the core of everything we build. We foster an open, inclusive environment where AI practitioners from diverse backgrounds collaborate to push the boundaries of machine learning.

- **Transparency & Collaboration:** Unlimited hosting of public models and datasets encourages knowledge sharing.
- **Learning & Growth:** Resources include comprehensive Docs, Tutorials, Daily Papers, active Discord and Forums, plus a rich Blog.
- **Open Source & Innovation:** Contributions and partnerships flourish through GitHub alongside ongoing community challenges.
- **Accessibility:** Building tools that democratize AI, making it accessible for enterprises, researchers, and hobbyists alike.

---

## Our Customers & Partners
Hugging Face serves a broad spectrum of users, including:
- **Academic researchers** seeking datasets and model benchmarks.
- **Enterprises** leveraging AI for innovation in sectors like healthcare, finance, and technology.
- **Developers & Data Scientists** building AI-powered products.
- Leading organizations and startups worldwide use Hugging Face to accelerate AI adoption through our enterprise solutions like Hugging Face PRO and tailored support.

---

## Career Opportunities
Joining Hugging Face means becoming part of an inspiring, fast-growing AI startup committed to shaping the future of machine learning. We value creativity, openness, and a collaborative spirit.

- Roles in machine learning research, engineering, product, community management, and more.
- Opportunities to work alongside world-class AI experts and contribute to open-source projects.
- A culture that encourages continuous learning, experimentation, and impact.

---

## Connect & Explore
Discover the future of AI with Hugging Face:
- Browse models & datasets: [huggingface.co/models](https://huggingface.co/models)
- Join the community: Discord & Forums
- Explore documentation & learning resources: [huggingface.co/docs](https://huggingface.co/docs)
- Follow latest news and blog posts: [huggingface.co/blog](https://huggingface.co/blog)

---

**Hugging Face** – Empowering the AI community to build the future, together.